In [ ]:
# =============================================================================
# Model 6 — LSTM (Long Short-Term Memory)
# Task    : Predict Crime_Count per LSOA based on sequences of past months
#
# Key difference from tabular models (Ridge, RF, XGBoost):
#   - Input is a 3D sequence tensor: (samples, timesteps, features)
#   - For each prediction, the model sees the last N months of ALL features
#   - Learns temporal dependencies across the sequence window
#
# Key difference from SARIMA:
#   - Uses ALL features (spatial, socioeconomic, weather, lags) not just crime count
#   - Learns non-linear temporal patterns across multiple features simultaneously
# =============================================================================

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

# --- Deep learning imports ---
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, BatchNormalization,
                                     Bidirectional, Input, Concatenate, Attention)
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv("./crime_per_capita_df.csv")
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

# =============================================================================
# 1.5 FEATURE ENGINEERING — Add crime-specific features
# =============================================================================
print("\nPerforming feature engineering...")

# Seasonal interactions with weather
df['temp_x_season'] = df['Max_Temperature_Celsius'] * df['is_summer'].astype(int)
df['rainfall_x_season'] = df['Rainfall_mm'] * df['is_winter'].astype(int)

# LSOA-specific crime volatility (helps identify hard-to-predict areas)
df['lsoa_crime_volatility'] = df.groupby('LSOA_Code')['Crime_Count'].transform('std').fillna(0)

# Socioeconomic interaction: deprivation × population density
df['deprivation_x_density'] = df['Income_Deprivation'] * df['pop_density']

# Recent crime trend (lag difference = momentum)
df['crime_trend'] = df.groupby('LSOA_Code')['crime_lag_1m'].transform('diff').fillna(0)

print(f"Added 5 new engineered features")

# =============================================================================
# 2. FEATURE SELECTION
# LSTM uses ALL features — including spatial and socioeconomic
# StandardScaler IS required (unlike tree models)
# Do NOT include Crime_Rate, Crime_per_Capita (derived from target)
# =============================================================================
FEATURES = [
    # --- Location ---
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",

    # --- Date ---
    "Year",
    "Month",
    "month_sin",
    "month_cos",

    # --- Season flags ---
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",

    # --- Socioeconomic ---
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",

    # --- Weather ---
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",

    # --- Lag & rolling (leak-free) ---
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
    
    # --- NEW: Engineered features ---
    "temp_x_season",
    "rainfall_x_season",
    "lsoa_crime_volatility",
    "deprivation_x_density",
    "crime_trend",
]

TARGET     = "Crime_Count"
SEQ_LENGTH = 12    # look back 12 months (1 full year) to predict next month

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

N_FEATURES = len(FEATURES)
print(f"\nFeatures       : {N_FEATURES}")
print(f"Sequence length: {SEQ_LENGTH} months lookback")

# =============================================================================
# 3. TIME-BASED TRAIN / TEST SPLIT
# =============================================================================
train_df = df[df["Year"].isin([2020, 2021, 2022, 2023])].copy()
test_df  = df[df["Year"] == 2024].copy()

print(f"Train rows : {len(train_df):,}  |  Test rows : {len(test_df):,}")

# =============================================================================
# 4. SCALING
# LSTM requires scaled inputs — fit scaler on train only, apply to both
# Scale features and target separately
# =============================================================================
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

# Fit on train
X_train_raw = feature_scaler.fit_transform(train_df[FEATURES])
y_train_raw = target_scaler.fit_transform(train_df[[TARGET]])

# Apply to test (use train statistics — no leakage)
X_test_raw  = feature_scaler.transform(test_df[FEATURES])
y_test_raw  = target_scaler.transform(test_df[[TARGET]])

# =============================================================================
# 5. SEQUENCE BUILDER
# Converts tabular rows into overlapping sequences per LSOA
#
# For each LSOA, for each month M:
#   X[i] = features of months [M-12, M-11, ..., M-1]  (shape: 12 x n_features)
#   y[i] = Crime_Count at month M
#
# This means the model sees a 12-month window of history to predict month 13
# =============================================================================
def build_sequences(data_df, scaled_X, scaled_y, seq_length, lsoa_col="LSOA_Code"):
    """
    Build (X_seq, y_seq, lsoa_ids, month_ids) for LSTM training/testing.
    data_df   : original dataframe (sorted by LSOA, Year, Month)
    scaled_X  : StandardScaler-transformed feature array (same row order as data_df)
    scaled_y  : StandardScaler-transformed target array
    seq_length: number of past months to include in each sequence
    """
    X_seqs, y_seqs, lsoas, months, years = [], [], [], [], []

    lsoa_list = data_df[lsoa_col].unique()

    for lsoa in lsoa_list:
        mask  = data_df[lsoa_col] == lsoa
        idx   = data_df[mask].index  # original row indices
        n_rows= len(idx)

        if n_rows <= seq_length:
            continue  # not enough history for this LSOA

        lsoa_X = scaled_X[mask.values]
        lsoa_y = scaled_y[mask.values]
        lsoa_months = data_df[mask]["Month"].values
        lsoa_years  = data_df[mask]["Year"].values

        for i in range(seq_length, n_rows):
            X_seqs.append(lsoa_X[i - seq_length : i])   # past seq_length months
            y_seqs.append(lsoa_y[i, 0])                  # target month
            lsoas.append(lsoa)
            months.append(lsoa_months[i])
            years.append(lsoa_years[i])

    return (np.array(X_seqs, dtype=np.float32),
            np.array(y_seqs, dtype=np.float32),
            np.array(lsoas),
            np.array(months),
            np.array(years))

print("\nBuilding training sequences...")
t0 = time.time()

# Build sequences from the FULL dataset sorted by LSOA + time
# For test: we need the last SEQ_LENGTH months of 2023 as context for 2024
full_df     = df.copy()
X_full_raw  = feature_scaler.transform(full_df[FEATURES])
y_full_raw  = target_scaler.transform(full_df[[TARGET]])

X_all, y_all, lsoa_all, month_all, year_all = build_sequences(
    full_df, X_full_raw, y_full_raw, SEQ_LENGTH
)

# Split sequences by year of the TARGET month
train_mask = year_all <= 2023
test_mask  = year_all == 2024

X_train_seq = X_all[train_mask]
y_train_seq = y_all[train_mask]
X_test_seq  = X_all[test_mask]
y_test_seq  = y_all[test_mask]
lsoa_test   = lsoa_all[test_mask]
month_test  = month_all[test_mask]

print(f"Sequence building : {time.time()-t0:.1f}s")
print(f"Train sequences   : {X_train_seq.shape}  → (samples, timesteps, features)")
print(f"Test sequences    : {X_test_seq.shape}")

# Ground truth (unscaled) for evaluation
y_test_true = target_scaler.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

# =============================================================================
# 6. EVALUATION HELPER
# =============================================================================
def evaluate(model_name, y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R²    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {"Model": model_name, "MAE": round(mae,4), "RMSE": round(rmse,4),
            "R2": round(r2,4), "MAPE": round(mape,2)}

# =============================================================================
# 7. SHARED TRAINING CALLBACKS
# EarlyStopping  : stop when val_loss stops improving (patience=15 epochs)
# ReduceLROnPlat : halve LR when stuck (patience=7 epochs)
# =============================================================================
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor="val_loss", patience=15,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=7, min_lr=1e-6, verbose=1
        ),
    ]

BATCH_SIZE  = 512    # large batch = faster training on 58GB RAM
EPOCHS      = 200    # early stopping will cut this short
VAL_SPLIT   = 0.15   # 15% of training sequences for validation

results = []

# =============================================================================
# 8. MODEL A — Simple LSTM (baseline)
# Single LSTM layer — fast to train, interpretable baseline
# =============================================================================
print("\n--- Model A: Simple LSTM ---")

def build_simple_lstm(seq_len, n_feat, units=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        LSTM(units, input_shape=(seq_len, n_feat), return_sequences=False),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_a = build_simple_lstm(SEQ_LENGTH, N_FEATURES)
model_a.summary()

t0 = time.time()
hist_a = model_a.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("simple_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_a.history['loss'])}")

pred_a_scaled = model_a.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_a = target_scaler.inverse_transform(pred_a_scaled).flatten()
res_a  = evaluate("LSTM — simple (64 units)", y_test_true, pred_a)
results.append(res_a)

# =============================================================================
# 9. MODEL B — Stacked LSTM
# Two LSTM layers — deeper temporal feature extraction
# return_sequences=True passes full sequence output to next LSTM layer
# =============================================================================
print("\n--- Model B: Stacked LSTM (2 layers) ---")

def build_stacked_lstm(seq_len, n_feat, units1=128, units2=64, dropout=0.3, lr=1e-3):
    model = Sequential([
        LSTM(units1, input_shape=(seq_len, n_feat), return_sequences=True),
        Dropout(dropout),
        LSTM(units2, return_sequences=False),
        Dropout(dropout),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_b = build_stacked_lstm(SEQ_LENGTH, N_FEATURES)
model_b.summary()

t0 = time.time()
hist_b = model_b.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("stacked_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_b.history['loss'])}")

pred_b_scaled = model_b.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_b = target_scaler.inverse_transform(pred_b_scaled).flatten()
res_b  = evaluate("LSTM — stacked (128→64 units)", y_test_true, pred_b)
results.append(res_b)

# =============================================================================
# 10. MODEL C — Bidirectional LSTM
# Reads sequence both forward and backward — captures future context within window
# Best for sequences where patterns depend on both preceding and following months
# =============================================================================
print("\n--- Model C: Bidirectional LSTM ---")

def build_bidir_lstm(seq_len, n_feat, units=96, dropout=0.3, lr=8e-4):
    model = Sequential([
        Bidirectional(LSTM(units, return_sequences=True),
                      input_shape=(seq_len, n_feat)),
        Dropout(dropout),
        Bidirectional(LSTM(48, return_sequences=False)),
        Dropout(dropout),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_c = build_bidir_lstm(SEQ_LENGTH, N_FEATURES)
model_c.summary()

t0 = time.time()
hist_c = model_c.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("bidir_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_c.history['loss'])}")

pred_c_scaled = model_c.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_c = target_scaler.inverse_transform(pred_c_scaled).flatten()
res_c  = evaluate("LSTM — bidirectional (96 units)", y_test_true, pred_c)
results.append(res_c)

# =============================================================================
# 11. MODEL D — Deep LSTM with L2 regularisation
# Wider + deeper with weight regularisation to handle the large feature set
# Best config for 58GB RAM — can afford larger batch and more units
# =============================================================================
print("\n--- Model D: Deep LSTM with regularisation ---")

def build_deep_lstm(seq_len, n_feat, lr=5e-4):
    model = Sequential([
        LSTM(256, input_shape=(seq_len, n_feat),
             return_sequences=True,
             kernel_regularizer=l2(1e-4),
             recurrent_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(128, return_sequences=True,
             kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(64, return_sequences=False),
        Dropout(0.2),

        Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_d = build_deep_lstm(SEQ_LENGTH, N_FEATURES)
model_d.summary()

t0 = time.time()
hist_d = model_d.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("deep_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_d.history['loss'])}")

pred_d_scaled = model_d.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_d = target_scaler.inverse_transform(pred_d_scaled).flatten()
res_d  = evaluate("LSTM — deep (256→128→64 units, L2 reg)", y_test_true, pred_d)
results.append(res_d)

# =============================================================================
# 12. MODEL E — LSTM with Attention + Huber Loss
# Attention layer weights each timestep — model focuses on relevant months
# Huber loss is robust to outliers and helps with high-crime predictions
# =============================================================================
print("\n--- Model E: LSTM with Attention + Huber Loss ---")

def build_attention_lstm(seq_len, n_feat, lr=5e-4):
    inputs = Input(shape=(seq_len, n_feat))
    
    # LSTM encoder
    lstm1 = LSTM(256, return_sequences=True, 
                 kernel_regularizer=l2(1e-4),
                 recurrent_regularizer=l2(1e-4))(inputs)
    lstm1 = BatchNormalization()(lstm1)
    lstm1 = Dropout(0.3)(lstm1)
    
    lstm2 = LSTM(128, return_sequences=True,
                 kernel_regularizer=l2(1e-4))(lstm1)
    lstm2 = BatchNormalization()(lstm2)
    lstm2 = Dropout(0.3)(lstm2)
    
    # Attention: Let model weight each timestep
    attn_out = Attention()([lstm2, lstm2])
    
    lstm3 = LSTM(64, return_sequences=False)(attn_out)
    lstm3 = Dropout(0.2)(lstm3)
    
    dense = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(lstm3)
    dense = BatchNormalization()(dense)
    dense = Dropout(0.2)(dense)
    dense = Dense(64, activation="relu")(dense)
    output = Dense(1)(dense)
    
    model = Model(inputs, output)
    # Use Huber loss instead of MSE for robustness
    model.compile(optimizer=Adam(lr), loss='huber', metrics=["mae"])
    return model

model_e = build_attention_lstm(SEQ_LENGTH, N_FEATURES)
model_e.summary()

t0 = time.time()
hist_e = model_e.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("attention_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_e.history['loss'])}")

pred_e_scaled = model_e.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_e = target_scaler.inverse_transform(pred_e_scaled).flatten()
res_e  = evaluate("LSTM — attention (256→128→64 units, Huber loss)", y_test_true, pred_e)
results.append(res_e)

# =============================================================================
# 13. PICK BEST LSTM MODEL
# =============================================================================
best      = min(results, key=lambda x: x["MAE"])
print(f"\n{'='*55}")
print(f"Best LSTM config : {best['Model']}")
print(f"MAE={best['MAE']}  RMSE={best['RMSE']}  R²={best['R2']}  MAPE={best['MAPE']}%")
print(f"{'='*55}")

# =============================================================================
# 14. ERROR ANALYSIS on best model
# =============================================================================
model_map = {
    "LSTM — simple (64 units)"               : pred_a,
    "LSTM — stacked (128→64 units)"          : pred_b,
    "LSTM — bidirectional (96 units)"        : pred_c,
    "LSTM — deep (256→128→64 units, L2 reg)" : pred_d,
    "LSTM — attention (256→128→64 units, Huber loss)" : pred_e,
}
best_preds = model_map[best["Model"]]

print("\n--- Error Analysis (best LSTM model) ---")
res_df = pd.DataFrame({
    "LSOA_Code" : lsoa_test,
    "Month"     : month_test,
    "Actual"    : y_test_true,
    "Predicted" : np.round(best_preds, 2),
    "Abs_Error" : np.abs(y_test_true - best_preds),
    "Pct_Error" : np.abs(y_test_true - best_preds) / (y_test_true + 1e-9) * 100,
})

# Error by crime bucket
res_df["Crime_Bucket"] = pd.cut(
    res_df["Actual"], bins=[0, 5, 15, 30, 50, 9999],
    labels=["0–5", "6–15", "16–30", "31–50", "50+"]
)
bucket_err = (res_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
              .agg(["mean", "count"])
              .rename(columns={"mean": "Avg_MAE", "count": "N_rows"}))
print("\nError by crime count bucket:")
print(bucket_err.to_string())

# Error by month
monthly = (res_df.groupby("Month")["Abs_Error"]
           .mean().reset_index()
           .rename(columns={"Abs_Error": "Avg_MAE"}))
print("\nAverage error by month:")
print(monthly.to_string(index=False))

# Worst LSOAs
worst = (res_df.groupby("LSOA_Code")["Abs_Error"]
         .mean().sort_values(ascending=False).head(10))
print("\nTop 10 hardest LSOAs:")
print(worst.to_string())

# Sample predictions
print("\n--- Sample Predictions vs Actual (first 12 rows) ---")
print(res_df.head(12)[
    ["LSOA_Code","Month","Actual","Predicted","Abs_Error"]
].to_string(index=False))

# =============================================================================
# 15. COMPARE ALL LSTM VARIANTS
# =============================================================================
print("\n--- All LSTM variants (including new improvements) ---")
lstm_summary = pd.DataFrame(results)
print(lstm_summary[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

# =============================================================================
# 16. SAVE TO MODEL COMPARISON FILE
# =============================================================================
new_rows = pd.DataFrame(results)

try:
    existing = pd.read_csv("model_comparison.csv")
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined.to_csv("model_comparison.csv", index=False)

print("\n\n--- Updated model_comparison.csv ---")
print(combined[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

# =============================================================================
# 17. SAVE BEST MODEL
# =============================================================================
best_model_map = {
    "LSTM — simple (64 units)"               : model_a,
    "LSTM — stacked (128→64 units)"          : model_b,
    "LSTM — bidirectional (96 units)"        : model_c,
    "LSTM — deep (256→128→64 units, L2 reg)" : model_d,
    "LSTM — attention (256→128→64 units, Huber loss)" : model_e,
}
best_model = best_model_map[best["Model"]]
best_model.save("best_lstm_model.keras")
print(f"\nBest LSTM model saved to best_lstm_model.keras")
print("(Load with: tf.keras.models.load_model('best_lstm_model.keras'))")

In [ ]:
# =============================================================================
# ADVANCED IMPROVEMENTS: Multi-Scale LSTM, Residual Connections, 
# Loss Engineering, and Hyperparameter Optimization
# =============================================================================

print("\n" + "="*70)
print("ADVANCED MODEL IMPROVEMENTS")
print("="*70)

# =============================================================================
# IMPROVEMENT 1: CNN-LSTM (Convolutional + Recurrent)
# Learns spatial patterns within each timestep sequence
# =============================================================================
print("\n--- Model F: CNN-LSTM (Spatial-Temporal Feature Extraction) ---")

from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, RepeatVector, TimeDistributed

def build_cnn_lstm(seq_len, n_feat, lr=5e-4):
    model = Sequential([
        # CNN to extract local temporal patterns
        Conv1D(64, kernel_size=3, activation="relu", 
               input_shape=(seq_len, n_feat), padding="same"),
        BatchNormalization(),
        Dropout(0.2),
        
        Conv1D(32, kernel_size=3, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        
        # LSTM on CNN features
        LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),
        
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        
        Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_f = build_cnn_lstm(SEQ_LENGTH, N_FEATURES)
model_f.summary()

t0 = time.time()
hist_f = model_f.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("cnn_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_f.history['loss'])}")

pred_f_scaled = model_f.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_f = target_scaler.inverse_transform(pred_f_scaled).flatten()
res_f = evaluate("LSTM — CNN-LSTM (Conv + 128→64 LSTM)", y_test_true, pred_f)
results.append(res_f)

# =============================================================================
# IMPROVEMENT 2: Multi-Scale Ensemble (6, 12, 24 month sequences)
# Different lookback windows capture different temporal patterns
# =============================================================================
print("\n" + "="*70)
print("--- Model G: Multi-Scale LSTM (Variable Lookback Windows) ---")
print("="*70)

def build_multiscale_sequences(data_df, scaled_X, scaled_y, seq_lengths, lsoa_col="LSOA_Code"):
    """Build sequences at multiple timesteps, returning year info for proper splitting"""
    all_seqs = {}
    
    for seq_len in seq_lengths:
        X_seqs, y_seqs, years_seqs = [], [], []
        lsoa_list = data_df[lsoa_col].unique()
        
        for lsoa in lsoa_list:
            mask = data_df[lsoa_col] == lsoa
            n_rows = len(data_df[mask])
            
            if n_rows <= seq_len:
                continue
                
            lsoa_X = scaled_X[mask.values]
            lsoa_y = scaled_y[mask.values]
            lsoa_years = data_df[mask]["Year"].values
            
            for i in range(seq_len, n_rows):
                X_seqs.append(lsoa_X[i - seq_len : i])
                y_seqs.append(lsoa_y[i, 0])
                years_seqs.append(lsoa_years[i])  # Year of TARGET month
        
        all_seqs[seq_len] = (np.array(X_seqs, dtype=np.float32),
                             np.array(y_seqs, dtype=np.float32),
                             np.array(years_seqs))
    
    return all_seqs

print("Building multi-scale sequences...")
seq_lengths = [6, 12, 24]
t0 = time.time()

multiscale_seqs = build_multiscale_sequences(full_df, X_full_raw, y_full_raw, seq_lengths)

pred_multiscale_all = []
multiscale_models = {}

for seq_len in seq_lengths:
    print(f"\n  Training on {seq_len}-month lookback...")
    X_seq, y_seq, years_seq = multiscale_seqs[seq_len]
    
    # Split by year: train on 2020-2023, test on 2024
    train_mask = years_seq <= 2023
    test_mask = years_seq == 2024
    
    X_tr = X_seq[train_mask]
    y_tr = y_seq[train_mask]
    X_ts = X_seq[test_mask]
    y_ts = y_seq[test_mask]
    
    print(f"    Train: {X_tr.shape[0]} sequences | Test: {X_ts.shape[0]} sequences")
    
    # Build simpler model for smaller sequences
    def build_multiscale_lstm(seq_len, n_feat):
        model = Sequential([
            LSTM(96, input_shape=(seq_len, n_feat), return_sequences=True,
                 kernel_regularizer=l2(5e-5)),
            Dropout(0.25),
            LSTM(48, return_sequences=False),
            Dropout(0.2),
            Dense(64, activation="relu"),
            Dense(1)
        ])
        model.compile(optimizer=Adam(3e-4), loss="mse", metrics=["mae"])
        return model
    
    model_ms = build_multiscale_lstm(seq_len, N_FEATURES)
    model_ms.fit(X_tr, y_tr, epochs=100, batch_size=512, 
                 validation_split=0.15, callbacks=get_callbacks("ms_lstm"),
                 verbose=0)
    
    # Store model for later use
    multiscale_models[seq_len] = model_ms
    
    # Predict on test if available
    if len(X_ts) > 0:
        pred_ms = model_ms.predict(X_ts, batch_size=BATCH_SIZE, verbose=0)
        pred_ms_unscaled = target_scaler.inverse_transform(pred_ms).flatten()
        pred_multiscale_all.append(pred_ms_unscaled)
        print(f"    Predictions: min={pred_ms_unscaled.min():.2f}, max={pred_ms_unscaled.max():.2f}")

print(f"\nMulti-scale training time: {(time.time()-t0)/60:.1f} min")

if pred_multiscale_all:
    # Need to align predictions to same length (use 12-month as reference)
    if len(pred_multiscale_all) > 1:
        # Pad shorter predictions or truncate longer ones to match X_test_seq length
        target_len = len(X_test_seq)
        pred_aligned = []
        
        for pred in pred_multiscale_all:
            if len(pred) == target_len:
                pred_aligned.append(pred)
            elif len(pred) < target_len:
                # Pad with mean values
                padded = np.pad(pred, (0, target_len - len(pred)), 
                               mode='constant', constant_values=pred.mean())
                pred_aligned.append(padded)
            else:
                # Truncate
                pred_aligned.append(pred[:target_len])
        
        pred_g = np.mean(pred_aligned, axis=0)
    else:
        # Only one scale available
        pred_g = np.pad(pred_multiscale_all[0], 
                       (0, max(0, len(X_test_seq) - len(pred_multiscale_all[0]))),
                       mode='constant', constant_values=pred_multiscale_all[0].mean())
    
    # Trim to actual test length
    pred_g = pred_g[:len(X_test_seq)]
    res_g = evaluate("LSTM — Multi-Scale Ensemble (6/12/24 months)", y_test_true, pred_g)
    results.append(res_g)

# =============================================================================
# IMPROVEMENT 3: Focal Loss (Focus on Hard Samples)
# Down-weight easy predictions, up-weight difficult ones
# =============================================================================
print("\n" + "="*70)
print("--- Model H: LSTM with Focal Loss (Focus on Outliers) ---")
print("="*70)

def focal_loss(gamma=2.0, alpha=0.25):
    """Focal loss for regression: focuses on hard samples"""
    def loss_fn(y_true, y_pred):
        # Calculate MAE residuals
        residual = tf.abs(y_true - y_pred)
        
        # Scale error by difficulty
        focal_weight = tf.pow(residual, gamma)
        
        # Combine with standard MAE
        return alpha * focal_weight * residual + (1 - alpha) * residual
    
    return loss_fn

def build_lstm_focal(seq_len, n_feat, lr=5e-4):
    model = Sequential([
        LSTM(192, input_shape=(seq_len, n_feat), return_sequences=True,
             kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),
        
        LSTM(96, return_sequences=False),
        Dropout(0.25),
        
        Dense(96, activation="relu", kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dense(48, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss=focal_loss(gamma=1.5), metrics=["mae"])
    return model

model_h = build_lstm_focal(SEQ_LENGTH, N_FEATURES)
model_h.summary()

t0 = time.time()
hist_h = model_h.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("focal_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_h.history['loss'])}")

pred_h_scaled = model_h.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_h = target_scaler.inverse_transform(pred_h_scaled).flatten()
res_h = evaluate("LSTM — Focal Loss (192→96 units)", y_test_true, pred_h)
results.append(res_h)

# =============================================================================
# IMPROVEMENT 4: Residual LSTM (Skip Connections)
# Improves gradient flow for deep networks
# =============================================================================
print("\n" + "="*70)
print("--- Model I: Residual LSTM (Skip Connections) ---")
print("="*70)

def build_residual_lstm(seq_len, n_feat, lr=5e-4):
    inputs = Input(shape=(seq_len, n_feat))
    
    # Initial projection
    x = Dense(128, activation="relu")(inputs)
    x = LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Residual block 1
    res_input = x
    x = LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = tf.keras.layers.Add()([x, res_input])  # Skip connection
    
    # Residual block 2
    res_input = x
    x = LSTM(64, return_sequences=True)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = tf.keras.layers.Add()([x, res_input])  # Skip connection
    
    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    
    x = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation="relu")(x)
    output = Dense(1)(x)
    
    model = Model(inputs, output)
    model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])
    return model

model_i = build_residual_lstm(SEQ_LENGTH, N_FEATURES)
model_i.summary()

t0 = time.time()
hist_i = model_i.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks("residual_lstm"),
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min | Epochs run: {len(hist_i.history['loss'])}")

pred_i_scaled = model_i.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0)
pred_i = target_scaler.inverse_transform(pred_i_scaled).flatten()
res_i = evaluate("LSTM — Residual (Skip Connections)", y_test_true, pred_i)
results.append(res_i)

# =============================================================================
# COMPARISON: Advanced Models vs Baseline
# =============================================================================
print("\n" + "="*70)
print("ADVANCED MODELS COMPARISON")
print("="*70)

advanced_results = pd.DataFrame(results[-4:])  # F, G, H, I
print("\nNew Advanced Models:")
print(advanced_results[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

# =============================================================================
# FINAL ENSEMBLE: All models (A-I) + Multiple Ensembles
# =============================================================================
print("\n" + "="*70)
print("ULTIMATE ENSEMBLE: All Models Combined")
print("="*70)

all_predictions = [pred_a, pred_b, pred_c, pred_d, pred_e, pred_f, pred_h, pred_i]
if 'pred_g' in globals() and pred_g is not None:
    all_predictions.append(pred_g)

# Simple average
ensemble_ultimate_avg = np.mean(all_predictions, axis=0)
res_ultimate_avg = evaluate("ENSEMBLE — All Models Average", y_test_true, ensemble_ultimate_avg)
results.append(res_ultimate_avg)

# Inverted MAE weighted
mae_all = [r["MAE"] for r in results[-len(all_predictions):]]
weights_all = np.array([1/m for m in mae_all])
weights_all = weights_all / np.sum(weights_all)

print(f"\nEnsemble weights (inverse MAE):")
for i, w in enumerate(weights_all):
    print(f"  Model {i+1}: {w:.4f}")

ensemble_ultimate_weighted = np.zeros_like(pred_a)
for i, pred in enumerate(all_predictions):
    ensemble_ultimate_weighted += pred * weights_all[i]

res_ultimate_weighted = evaluate("ENSEMBLE — Ultimate Weighted", y_test_true, ensemble_ultimate_weighted)
results.append(res_ultimate_weighted)

# =============================================================================
# FINAL SUMMARY & EXPORT
# =============================================================================
print("\n" + "="*70)
print("FINAL COMPREHENSIVE RESULTS (All Models)")
print("="*70)

final_all_results = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
print("\n" + final_all_results[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

# Save comprehensive results
final_all_results.to_csv("lstm_improvements_comprehensive.csv", index=False)
print("\n✓ Comprehensive results saved to lstm_improvements_comprehensive.csv")

# Identify best
best_idx = final_all_results["MAE"].idxmin()
best_improvement = final_all_results.iloc[best_idx]
print(f"\n🏆 BEST MODEL: {best_improvement['Model']}")
print(f"   MAE={best_improvement['MAE']:.4f}  RMSE={best_improvement['RMSE']:.4f}")
print(f"   R²={best_improvement['R2']:.4f}  MAPE={best_improvement['MAPE']:.2f}%")

# Compare to original best
original_best_mae = 5.1272  # Deep LSTM
improvement_pct = ((original_best_mae - best_improvement['MAE']) / original_best_mae) * 100
print(f"\n📈 Improvement over original Deep LSTM: {improvement_pct:.1f}%")

print("\n✓ All advanced models trained and evaluated!")

In [ ]:
# =============================================================================
# MODEL 6 - LSTM (Long Short-Term Memory)
# Task: Predict Crime_Count per LSOA based on sequences of past months
# =============================================================================

import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Add,
    Attention,
    BatchNormalization,
    Bidirectional,
    Conv1D,
    Dense,
    Dropout,
    Input,
    LSTM,
    MaxPooling1D,
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "./crime_per_capita_df.csv"
TARGET = "Crime_Count"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR = 2024
SEQ_LENGTH = 12

BATCH_SIZE = 512
EPOCHS = 200
VAL_SPLIT = 0.15

HOTSPOT_PERCENTS = [5, 10, 20]

MODEL_COMPARISON_PATH = "model_comparison.csv"
FINAL_RESULTS_PATH = "model_comparison_final.csv"
COMPREHENSIVE_RESULTS_PATH = "lstm_improvements_comprehensive.csv"
BEST_MODEL_PATH = "lstm_model.keras"
REPORT_TXT_PATH = "evaluation_results_lstm.txt"
ARTIFACT_PATH = Path("artifacts") / "lstm_model.joblib"

FEATURES = [
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",
    "Year",
    "Month",
    "month_sin",
    "month_cos",
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
    "temp_x_season",
    "rainfall_x_season",
    "lsoa_crime_volatility",
    "deprivation_x_density",
    "crime_trend",
]

# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def evaluate(model_name, y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 60}")
    print(f"  {model_name}")
    print(f"{'=' * 60}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R2    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai  # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(
    output_path,
    final_results_df,
    hotspot_df,
    best_model_name,
    exported_model_name,
    best_model_file,
    artifact_file,
):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI — Prediction Accuracy Index\n")
        f.write("# RRI — Recapture Rate Index\n")
        f.write("# PEI — Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")
        f.write("# =============================================================================\n")
        f.write("# LSTM RESULTS SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(final_results_df[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hotspot_df.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Best overall model          : {best_model_name}\n")
        f.write(f"Exported Keras model        : {exported_model_name}\n")
        f.write(f"Keras file                  : {best_model_file}\n")
        f.write(f"Metadata artifact           : {artifact_file}\n")


def get_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def build_sequences(data_df, scaled_X, scaled_y, seq_length, lsoa_col="LSOA_Code"):
    X_seqs, y_seqs, lsoas, months, years = [], [], [], [], []
    lsoa_values = data_df[lsoa_col].unique()

    for lsoa in lsoa_values:
        mask = data_df[lsoa_col] == lsoa
        n_rows = int(mask.sum())

        if n_rows <= seq_length:
            continue

        lsoa_X = scaled_X[mask.values]
        lsoa_y = scaled_y[mask.values]
        lsoa_months = data_df.loc[mask, "Month"].values
        lsoa_years = data_df.loc[mask, "Year"].values

        for i in range(seq_length, n_rows):
            X_seqs.append(lsoa_X[i - seq_length : i])
            y_seqs.append(lsoa_y[i, 0])
            lsoas.append(lsoa)
            months.append(int(lsoa_months[i]))
            years.append(int(lsoa_years[i]))

    return (
        np.array(X_seqs, dtype=np.float32),
        np.array(y_seqs, dtype=np.float32),
        np.array(lsoas),
        np.array(months),
        np.array(years),
    )


def build_multiscale_sequences(data_df, scaled_X, scaled_y, seq_lengths, lsoa_col="LSOA_Code"):
    all_seqs = {}
    for seq_len in seq_lengths:
        X_seqs, y_seqs, years_seqs = [], [], []
        for lsoa in data_df[lsoa_col].unique():
            mask = data_df[lsoa_col] == lsoa
            n_rows = int(mask.sum())
            if n_rows <= seq_len:
                continue

            lsoa_X = scaled_X[mask.values]
            lsoa_y = scaled_y[mask.values]
            lsoa_years = data_df.loc[mask, "Year"].values

            for i in range(seq_len, n_rows):
                X_seqs.append(lsoa_X[i - seq_len : i])
                y_seqs.append(lsoa_y[i, 0])
                years_seqs.append(int(lsoa_years[i]))

        all_seqs[seq_len] = (
            np.array(X_seqs, dtype=np.float32),
            np.array(y_seqs, dtype=np.float32),
            np.array(years_seqs),
        )
    return all_seqs


def align_prediction(pred, target_len):
    pred = np.asarray(pred, dtype=float).flatten()
    if len(pred) == target_len:
        return pred
    if len(pred) == 0:
        return np.zeros(target_len, dtype=float)
    if len(pred) < target_len:
        return np.pad(pred, (0, target_len - len(pred)), mode="constant", constant_values=float(np.mean(pred)))
    return pred[:target_len]


def train_and_predict(
    model_name,
    model_obj,
    X_tr,
    y_tr,
    X_ts,
    y_true_unscaled,
    target_scaler_obj,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    verbose=1,
):
    t0 = time.time()
    history = model_obj.fit(
        X_tr,
        y_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=val_split,
        callbacks=get_callbacks(),
        verbose=verbose,
    )
    print(f"Training time: {(time.time() - t0) / 60:.1f} min | Epochs run: {len(history.history['loss'])}")

    pred_scaled = model_obj.predict(X_ts, batch_size=batch_size, verbose=0)
    pred_unscaled = target_scaler_obj.inverse_transform(pred_scaled).flatten()

    result = evaluate(model_name, y_true_unscaled, pred_unscaled)
    return result, pred_unscaled, history


# =============================================================================
# 3. MODEL BUILDERS
# =============================================================================
def build_simple_lstm(seq_len, n_feat, units=64, dropout=0.2, lr=1e-3):
    model = Sequential(
        [
            LSTM(units, input_shape=(seq_len, n_feat), return_sequences=False),
            Dropout(dropout),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def build_stacked_lstm(seq_len, n_feat, units1=128, units2=64, dropout=0.3, lr=1e-3):
    model = Sequential(
        [
            LSTM(units1, input_shape=(seq_len, n_feat), return_sequences=True),
            Dropout(dropout),
            LSTM(units2, return_sequences=False),
            Dropout(dropout),
            Dense(64, activation="relu"),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def build_bidir_lstm(seq_len, n_feat, units=96, dropout=0.3, lr=8e-4):
    model = Sequential(
        [
            Bidirectional(LSTM(units, return_sequences=True), input_shape=(seq_len, n_feat)),
            Dropout(dropout),
            Bidirectional(LSTM(48, return_sequences=False)),
            Dropout(dropout),
            Dense(64, activation="relu"),
            BatchNormalization(),
            Dense(32, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def build_deep_lstm(seq_len, n_feat, lr=5e-4):
    model = Sequential(
        [
            LSTM(
                256,
                input_shape=(seq_len, n_feat),
                return_sequences=True,
                kernel_regularizer=l2(1e-4),
                recurrent_regularizer=l2(1e-4),
            ),
            BatchNormalization(),
            Dropout(0.3),
            LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dropout(0.3),
            LSTM(64, return_sequences=False),
            Dropout(0.2),
            Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dropout(0.2),
            Dense(64, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def build_attention_lstm(seq_len, n_feat, lr=5e-4):
    inputs = Input(shape=(seq_len, n_feat))

    x = LSTM(
        256,
        return_sequences=True,
        kernel_regularizer=l2(1e-4),
        recurrent_regularizer=l2(1e-4),
    )(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Attention()([x, x])

    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.2)(x)

    x = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation="relu")(x)
    out = Dense(1)(x)

    model = Model(inputs, out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="huber", metrics=["mae"])
    return model


def build_cnn_lstm(seq_len, n_feat, lr=5e-4):
    model = Sequential(
        [
            Conv1D(64, kernel_size=3, activation="relu", input_shape=(seq_len, n_feat), padding="same"),
            BatchNormalization(),
            Dropout(0.2),
            Conv1D(32, kernel_size=3, activation="relu", padding="same"),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),
            LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dropout(0.3),
            LSTM(64, return_sequences=False),
            Dropout(0.2),
            Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dense(64, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


def build_multiscale_lstm(seq_len, n_feat):
    model = Sequential(
        [
            LSTM(96, input_shape=(seq_len, n_feat), return_sequences=True, kernel_regularizer=l2(5e-5)),
            Dropout(0.25),
            LSTM(48, return_sequences=False),
            Dropout(0.2),
            Dense(64, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=3e-4), loss="mse", metrics=["mae"])
    return model


def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        residual = tf.abs(y_true - y_pred)
        focal_weight = tf.pow(residual, gamma)
        return alpha * focal_weight * residual + (1 - alpha) * residual

    return loss_fn


def build_lstm_focal(seq_len, n_feat, lr=5e-4):
    model = Sequential(
        [
            LSTM(192, input_shape=(seq_len, n_feat), return_sequences=True, kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dropout(0.3),
            LSTM(96, return_sequences=False),
            Dropout(0.25),
            Dense(96, activation="relu", kernel_regularizer=l2(1e-4)),
            BatchNormalization(),
            Dense(48, activation="relu"),
            Dense(1),
        ]
    )
    model.compile(optimizer=Adam(learning_rate=lr), loss=focal_loss(gamma=1.5), metrics=["mae"])
    return model


def build_residual_lstm(seq_len, n_feat, lr=5e-4):
    inputs = Input(shape=(seq_len, n_feat))

    x = Dense(128, activation="relu")(inputs)
    x = LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    res_1 = x
    x = LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Add()([x, res_1])

    res_2 = x
    x = LSTM(64, return_sequences=True)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    res_2_proj = Dense(64)(res_2)
    x = Add()([x, res_2_proj])

    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation="relu")(x)
    out = Dense(1)(x)

    model = Model(inputs, out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


# =============================================================================
# 4. LOAD DATA + FEATURE ENGINEERING
# =============================================================================
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

print("\nPerforming feature engineering...")
df["temp_x_season"] = df["Max_Temperature_Celsius"] * df["is_summer"].astype(int)
df["rainfall_x_season"] = df["Rainfall_mm"] * df["is_winter"].astype(int)
df["lsoa_crime_volatility"] = df.groupby("LSOA_Code")["Crime_Count"].transform("std").fillna(0)
df["deprivation_x_density"] = df["Income_Deprivation"] * df["pop_density"]
df["crime_trend"] = df.groupby("LSOA_Code")["crime_lag_1m"].transform("diff").fillna(0)

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING - missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

N_FEATURES = len(FEATURES)
print(f"Added engineered features")
print(f"Features       : {N_FEATURES}")
print(f"Sequence length: {SEQ_LENGTH} months")

# =============================================================================
# 5. TRAIN/TEST SPLIT + SCALING
# =============================================================================
train_df = df[df["Year"].isin(TRAIN_YEARS)].copy()
test_df = df[df["Year"] == TEST_YEAR].copy()
print(f"Train rows : {len(train_df):,}  |  Test rows : {len(test_df):,}")

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_raw = feature_scaler.fit_transform(train_df[FEATURES])
y_train_raw = target_scaler.fit_transform(train_df[[TARGET]])

# Not directly used for training; kept for sanity checks
X_test_raw = feature_scaler.transform(test_df[FEATURES])
y_test_raw = target_scaler.transform(test_df[[TARGET]])

# =============================================================================
# 6. BUILD SEQUENCES
# =============================================================================
print("\nBuilding sequences...")
t0 = time.time()

full_df = df.copy()
X_full_raw = feature_scaler.transform(full_df[FEATURES])
y_full_raw = target_scaler.transform(full_df[[TARGET]])

X_all, y_all, lsoa_all, month_all, year_all = build_sequences(
    full_df, X_full_raw, y_full_raw, SEQ_LENGTH
)

train_mask = year_all <= 2023
test_mask = year_all == 2024

X_train_seq = X_all[train_mask]
y_train_seq = y_all[train_mask]
X_test_seq = X_all[test_mask]
y_test_seq = y_all[test_mask]
lsoa_test = lsoa_all[test_mask]
month_test = month_all[test_mask]

print(f"Sequence building : {time.time() - t0:.1f}s")
print(f"Train sequences   : {X_train_seq.shape}")
print(f"Test sequences    : {X_test_seq.shape}")

y_test_true = target_scaler.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

# =============================================================================
# 7. TRAIN BASE MODELS (A-E)
# =============================================================================
results = []
predictions = {}
trained_models = {}

base_model_specs = [
    ("LSTM - simple (64 units)", build_simple_lstm(SEQ_LENGTH, N_FEATURES)),
    ("LSTM - stacked (128->64 units)", build_stacked_lstm(SEQ_LENGTH, N_FEATURES)),
    ("LSTM - bidirectional (96 units)", build_bidir_lstm(SEQ_LENGTH, N_FEATURES)),
    ("LSTM - deep (256->128->64 units, L2)", build_deep_lstm(SEQ_LENGTH, N_FEATURES)),
    ("LSTM - attention (256->128->64 units, Huber)", build_attention_lstm(SEQ_LENGTH, N_FEATURES)),
]

for model_name, model_obj in base_model_specs:
    print(f"\n--- Training {model_name} ---")
    res, pred, _ = train_and_predict(
        model_name=model_name,
        model_obj=model_obj,
        X_tr=X_train_seq,
        y_tr=y_train_seq,
        X_ts=X_test_seq,
        y_true_unscaled=y_test_true,
        target_scaler_obj=target_scaler,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        val_split=VAL_SPLIT,
        verbose=1,
    )
    results.append(res)
    predictions[model_name] = pred
    trained_models[model_name] = model_obj

# =============================================================================
# 8. ADVANCED MODEL F - CNN-LSTM
# =============================================================================
print("\n--- Training LSTM - CNN-LSTM ---")
model_f = build_cnn_lstm(SEQ_LENGTH, N_FEATURES)
res_f, pred_f, _ = train_and_predict(
    model_name="LSTM - CNN-LSTM (Conv + 128->64)",
    model_obj=model_f,
    X_tr=X_train_seq,
    y_tr=y_train_seq,
    X_ts=X_test_seq,
    y_true_unscaled=y_test_true,
    target_scaler_obj=target_scaler,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    verbose=1,
)
results.append(res_f)
predictions[res_f["Model"]] = pred_f
trained_models[res_f["Model"]] = model_f

# =============================================================================
# 9. ADVANCED MODEL G - MULTI-SCALE ENSEMBLE (6/12/24)
# =============================================================================
print("\n--- Training LSTM - multi-scale ensemble (6/12/24) ---")
seq_lengths = [6, 12, 24]
multiscale_seqs = build_multiscale_sequences(full_df, X_full_raw, y_full_raw, seq_lengths)

pred_multiscale_all = []
for seq_len in seq_lengths:
    X_seq, y_seq, years_seq = multiscale_seqs[seq_len]
    if len(X_seq) == 0:
        continue

    tr_mask = years_seq <= 2023
    ts_mask = years_seq == 2024
    X_tr = X_seq[tr_mask]
    y_tr = y_seq[tr_mask]
    X_ts = X_seq[ts_mask]

    print(f"  seq={seq_len}: train={X_tr.shape[0]} | test={X_ts.shape[0]}")
    if len(X_tr) == 0 or len(X_ts) == 0:
        continue

    ms_model = build_multiscale_lstm(seq_len, N_FEATURES)
    ms_model.fit(
        X_tr,
        y_tr,
        epochs=100,
        batch_size=512,
        validation_split=0.15,
        callbacks=get_callbacks(),
        verbose=0,
    )
    pred_ms_scaled = ms_model.predict(X_ts, batch_size=BATCH_SIZE, verbose=0)
    pred_ms = target_scaler.inverse_transform(pred_ms_scaled).flatten()
    pred_multiscale_all.append(align_prediction(pred_ms, len(X_test_seq)))

if pred_multiscale_all:
    pred_g = np.mean(np.vstack(pred_multiscale_all), axis=0)
    res_g = evaluate("LSTM - multi-scale ensemble (6/12/24 months)", y_test_true, pred_g)
    results.append(res_g)
    predictions[res_g["Model"]] = pred_g
else:
    print("Skipping multi-scale result: no valid test predictions produced.")

# =============================================================================
# 10. ADVANCED MODEL H - FOCAL LOSS
# =============================================================================
print("\n--- Training LSTM - focal loss ---")
model_h = build_lstm_focal(SEQ_LENGTH, N_FEATURES)
res_h, pred_h, _ = train_and_predict(
    model_name="LSTM - focal loss (192->96 units)",
    model_obj=model_h,
    X_tr=X_train_seq,
    y_tr=y_train_seq,
    X_ts=X_test_seq,
    y_true_unscaled=y_test_true,
    target_scaler_obj=target_scaler,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    verbose=1,
)
results.append(res_h)
predictions[res_h["Model"]] = pred_h
trained_models[res_h["Model"]] = model_h

# =============================================================================
# 11. ADVANCED MODEL I - RESIDUAL LSTM
# =============================================================================
print("\n--- Training LSTM - residual skip-connections ---")
model_i = build_residual_lstm(SEQ_LENGTH, N_FEATURES)
res_i, pred_i, _ = train_and_predict(
    model_name="LSTM - residual skip-connections",
    model_obj=model_i,
    X_tr=X_train_seq,
    y_tr=y_train_seq,
    X_ts=X_test_seq,
    y_true_unscaled=y_test_true,
    target_scaler_obj=target_scaler,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    verbose=1,
)
results.append(res_i)
predictions[res_i["Model"]] = pred_i
trained_models[res_i["Model"]] = model_i

# =============================================================================
# 12. ENSEMBLES
# =============================================================================
print("\n--- Building ensembles ---")
individual_model_names = list(predictions.keys())
stack_preds = np.vstack([predictions[name] for name in individual_model_names])

# Average ensemble
pred_avg = np.mean(stack_preds, axis=0)
res_avg = evaluate("ENSEMBLE - all models average", y_test_true, pred_avg)
results.append(res_avg)
predictions[res_avg["Model"]] = pred_avg

# Weighted inverse-MAE ensemble
mae_lookup = {r["Model"]: r["MAE"] for r in results}
weights = np.array([1.0 / max(mae_lookup[name], 1e-9) for name in individual_model_names], dtype=float)
weights = weights / weights.sum()
pred_weighted = np.average(stack_preds, axis=0, weights=weights)

print("\nInverse-MAE ensemble weights:")
for i, name in enumerate(individual_model_names):
    print(f"  {name}: {weights[i]:.4f}")

res_weighted = evaluate("ENSEMBLE - weighted inverse-MAE", y_test_true, pred_weighted)
results.append(res_weighted)
predictions[res_weighted["Model"]] = pred_weighted

# Median ensemble
pred_median = np.median(stack_preds, axis=0)
res_median = evaluate("ENSEMBLE - median", y_test_true, pred_median)
results.append(res_median)
predictions[res_median["Model"]] = pred_median

# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================
final_all_results = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)

print("\n" + "=" * 70)
print("FINAL COMPREHENSIVE RESULTS (All Models)")
print("=" * 70)
print(final_all_results[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

best_row = final_all_results.iloc[0]
best_model_name = best_row["Model"]
best_preds = predictions[best_model_name]

print(f"\nBest overall model: {best_model_name}")
print(f"MAE={best_row['MAE']:.4f}  RMSE={best_row['RMSE']:.4f}  R2={best_row['R2']:.4f}  MAPE={best_row['MAPE']:.2f}%")

# Save full LSTM result tables
final_all_results.to_csv(COMPREHENSIVE_RESULTS_PATH, index=False)
final_all_results.to_csv(FINAL_RESULTS_PATH, index=False)
print(f"\nSaved: {COMPREHENSIVE_RESULTS_PATH}")
print(f"Saved: {FINAL_RESULTS_PATH}")

# Append/update model_comparison.csv
new_rows = final_all_results.copy()
try:
    existing = pd.read_csv(MODEL_COMPARISON_PATH)
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined = combined.drop_duplicates(subset=["Model"], keep="last").reset_index(drop=True)
combined.to_csv(MODEL_COMPARISON_PATH, index=False)

print(f"\nUpdated: {MODEL_COMPARISON_PATH}")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

# =============================================================================
# 14. ERROR ANALYSIS ON BEST OVERALL MODEL
# =============================================================================
print("\n--- Error analysis (best overall) ---")
res_df = pd.DataFrame(
    {
        "LSOA_Code": lsoa_test,
        "Month": month_test,
        "Actual": y_test_true,
        "Predicted": np.round(best_preds, 2),
        "Abs_Error": np.abs(y_test_true - best_preds),
        "Pct_Error": np.abs(y_test_true - best_preds) / (y_test_true + 1e-9) * 100,
    }
)

res_df["Crime_Bucket"] = pd.cut(
    res_df["Actual"],
    bins=[0, 5, 15, 30, 50, 9999],
    labels=["0-5", "6-15", "16-30", "31-50", "50+"],
)

bucket_err = (
    res_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Avg_MAE", "count": "N_rows"})
)
print("\nError by crime count bucket:")
print(bucket_err.to_string())

monthly_err = (
    res_df.groupby("Month")["Abs_Error"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Error": "Avg_MAE"})
)
print("\nAverage error by month:")
print(monthly_err.to_string(index=False))

worst = (
    res_df.groupby("LSOA_Code")["Abs_Error"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop 10 hardest LSOAs:")
print(worst.to_string())

print("\n--- Sample predictions vs actual (first 12 rows) ---")
print(res_df.head(12)[["LSOA_Code", "Month", "Actual", "Predicted", "Abs_Error"]].to_string(index=False))

# =============================================================================
# 15. HOTSPOT EVALUATION ON BEST OVERALL MODEL
# =============================================================================
print("\n--- Crime Forecasting Indices ---")
hotspot_results = []
for pct in HOTSPOT_PERCENTS:
    hs = crime_hotspot_metrics(
        y_true=y_test_true,
        y_pred=best_preds,
        hotspot_percent=pct,
        model_name=best_model_name,
    )
    hotspot_results.append(hs)

hotspot_df = pd.DataFrame(hotspot_results)
print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

# =============================================================================
# 16. EXPORT BEST MODEL (.keras) + METADATA ARTIFACT
# =============================================================================
# If ensemble wins, export best individual neural network model for deployment.
if best_model_name in trained_models:
    export_model_name = best_model_name
else:
    individual_rows = final_all_results[final_all_results["Model"].isin(trained_models.keys())]
    export_model_name = individual_rows.iloc[0]["Model"]

export_model_obj = trained_models[export_model_name]
export_model_obj.save(BEST_MODEL_PATH)
print(f"\nSaved Keras model: {BEST_MODEL_PATH} (from: {export_model_name})")

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
artifact_payload = {
    "best_overall_model_name": best_model_name,
    "exported_keras_model_name": export_model_name,
    "best_model_file": BEST_MODEL_PATH,
    "features": FEATURES,
    "target": TARGET,
    "sequence_length": SEQ_LENGTH,
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "val_split": VAL_SPLIT,
    "hotspot_percents": HOTSPOT_PERCENTS,
    "best_metrics": {
        "MAE": float(best_row["MAE"]),
        "RMSE": float(best_row["RMSE"]),
        "R2": float(best_row["R2"]),
        "MAPE": float(best_row["MAPE"]),
    },
    "feature_scaler": feature_scaler,
    "target_scaler": target_scaler,
}

joblib.dump(artifact_payload, ARTIFACT_PATH)
print(f"Saved metadata artifact: {ARTIFACT_PATH}")

# =============================================================================
# 17. WRITE SINGLE TEXT REPORT (OVERWRITE EACH RUN)
# =============================================================================
write_evaluation_report(
    output_path=REPORT_TXT_PATH,
    final_results_df=final_all_results,
    hotspot_df=hotspot_df,
    best_model_name=best_model_name,
    exported_model_name=export_model_name,
    best_model_file=BEST_MODEL_PATH,
    artifact_file=str(ARTIFACT_PATH),
)
print(f"\nSaved evaluation text report: {REPORT_TXT_PATH}")
print("\nAll training, evaluation, hotspot metrics, and exports completed.")

I0000 00:00:1775035557.520497   29120 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775035579.292043   29120 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775035593.610502   29120 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version : 2.21.0
GPU available      : False


E0000 00:00:1775035597.381553   29120 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



Dataset shape : (304336, 39)
Date range    : 2020 to 2024
Unique LSOAs  : 12415

Performing feature engineering...
Added engineered features
Features       : 35
Sequence length: 12 months
Train rows : 241,710  |  Test rows : 62,626

Building sequences...
Sequence building : 297.2s
Train sequences   : (169956, 12, 35)
Test sequences    : (60008, 12, 35)

--- Training LSTM - simple (64 units) ---
Epoch 1/200


W0000 00:00:1775035899.592974   29120 cpu_allocator_impl.cc:82] Allocation of 242696160 exceeds 10% of free system memory.


...


  LSTM - simple (64 units)
  MAE   : 5.4472
  RMSE  : 20.7802
  R2    : 0.6793
  MAPE  : 53.75%

--- Training LSTM - stacked (128->64 units) ---
Epoch 1/200


W0000 00:00:1775036401.783970   29120 cpu_allocator_impl.cc:82] Allocation of 242696160 exceeds 10% of free system memory.


282/283 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.1428 - mae: 0.2190

W0000 00:00:1775036423.754746   29120 cpu_allocator_impl.cc:82] Allocation of 42829920 exceeds 10% of free system memory.


283/283 ━━━━━━━━━━━━━━━━━━━━ 23s 74ms/step - loss: 0.1026 - mae: 0.1982 - val_loss: 2.5077 - val_mae: 0.3382 - learning_rate: 0.0010
Epoch 2/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - loss: 0.0772 - mae: 0.1849 - val_loss: 2.2695 - val_mae: 0.3326 - learning_rate: 0.0010
Epoch 3/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 21s 73ms/step - loss: 0.0739 - mae: 0.1831 - val_loss: 2.1638 - val_mae: 0.3193 - learning_rate: 0.0010
Epoch 4/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 20s 69ms/step - loss: 0.0739 - mae: 0.1817 - val_loss: 2.3602 - val_mae: 0.3304 - learning_rate: 0.0010
Epoch 5/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 20s 69ms/step - loss: 0.0705 - mae: 0.1803 - val_loss: 2.1915 - val_mae: 0.3257 - learning_rate: 0.0010
Epoch 6/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - loss: 0.0694 - mae: 0.1789 - val_loss: 2.3125 - val_mae: 0.3305 - learning_rate: 0.0010
Epoch 7/200
283/283 ━━━━━━━━━━━━━━━━━━━━ 19s 68ms/step - loss: 0.0679 - mae: 0.1783 - val_loss: 2.0616 - val_mae: 0.3186 - learning_rate: 0.00